In [1]:
import rawpy  # pip install rawpy
import cv2 # pip install opencv-python + pip install opencv-contrib-python
import glob
import math
import numpy as np
import os
from skimage.color import rgb2lab # pip3 install scikit-image
from PIL import Image, ImageCms
from colormath.color_objects import AdobeRGBColor, LabColor #pip3 install colormath // requires setuptools+networkx
from colormath.color_conversions import convert_color
import pandas as pd
from scipy import stats
import time
import matplotlib.pyplot as plt

In [ ]:
#DETECTION - SENSOR APPROACH - Coef_S iterative


import matplotlib.pyplot as plt

folder = "Gamma_1"

px = [420, 550, 690, 850]
y = 850
winsz = 10
color_roi = (255,0,0)



files = [folder+"/papers_colors_v3_small_scene1_0347.tif", folder+"/papers_colors_v3_small_scene2_0354.tif", folder+"/papers_colors_v3_small_scene3_0364.tif", folder+"/papers_colors_v3_small_scene4_0371.tif"]

df = pd.DataFrame() 
idx = 0



for f in files:
    
    if "coef" in f or "GT" in f: continue
            
    print ("converting to HSV: "+f)
    
    img = cv2.imread(f, -1)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    size = img.shape
    img_hsv = np.zeros((size[0], size[1], 3))
    
    
    for i in range (img.shape[0]):
        for j in range (img.shape[1]):
            R = ((img_rgb[i,j,0])/65536)*255
            G = ((img_rgb[i,j,1])/65536)*255
            B = ((img_rgb[i,j,2])/65536)*255
            
            RGB = (R,G,B)
            
            maxi = max(RGB)
            mini = min(RGB)
            
            if maxi == mini:
                mini = maxi-1
            
            if maxi == RGB[0]:
                H = 60*((G-B)/(maxi-mini))
            elif maxi == RGB[1]:
                H = 120+60*((B-R)/(maxi-mini))
            elif maxi == RGB[2]:
                H = 240+60*((R-G)/(maxi-mini)) 
                
            if maxi == 0:
                S = 0
            else:
                S = 100*(float((maxi-mini)/maxi))
            
            V = (maxi/255)*100  
            
            img_hsv[i,j,2] = V
            img_hsv[i,j,1] = S
            img_hsv[i,j,0] = H
            
        
    for x in px:
        
        if x == px[0]: pap_x = 1
        elif x == px[1]: pap_x = 2
        elif x == px[2]: pap_x = 3
        elif x == px[3]: pap_x = 4
        
        img_vis = (img/256).astype('uint8')
        img_roi = cv2.rectangle(img_vis, (y-(winsz//2), x-(winsz//2)), (y+winsz, x+winsz), color_roi, 3)
        img_roi_rgb = cv2.cvtColor(img_roi, cv2.COLOR_BGR2RGB)

        
        h_roi_np = img_hsv[(x-(winsz//2)):(x+(winsz//2)),(y-(winsz//2)):(y+(winsz//2)),0]

        h_roi = h_roi_np.flatten()
        h_mean = np.mean(h_roi)
        h_std = np.std(h_roi)

        
        k = 3.16  
        outlier_indices = np.where((h_roi > (h_mean+k*h_std)) | (h_roi < (h_mean-k*h_std)))[0]
        print ("H outliers:",  outlier_indices)
        h_no_out = np.delete(h_roi, outlier_indices)

        h = np.mean(h_no_out)
        min_h = np.min(h_no_out)
        max_h = np.max(h_no_out)
        h_noise = max_h-min_h   
        print ("H: ", h)
        print ("H range in roi: ", min_h,     max_h)
        print ("H noise: ", h_noise)
        
        
        s_roi_np = img_hsv[(x-(winsz//2)):(x+(winsz//2)),(y-(winsz//2)):(y+(winsz//2)),1]
        
        #Descartar outliers amb mètode Chevyshev
        s_roi = s_roi_np.flatten()
        s_mean = np.mean(s_roi)
        s_std = np.std(s_roi)

        
        k = 3.16  
        outlier_indices = np.where((s_roi > (s_mean+k*s_std)) | (s_roi < (s_mean-k*s_std)))[0]
        print ("S outliers:",  outlier_indices)
        s_no_out = np.delete(s_roi, outlier_indices)
        
        s = np.mean(s_no_out)
        std_s = np.std(s_no_out)
        min_s = np.min(s_no_out)
        max_s = np.max(s_no_out)
        s_noise = max_s-min_s  
        print ("S: ", s)
        print ("S range in roi: ", min_s,     max_s)
        print ("S noise: ", s_noise)

        v_roi = img_hsv[(x-(winsz//2)):(x+(winsz//2)),(y-(winsz//2)):(y+(winsz//2)),2]
        v = np.mean(v_roi)
        std_v = np.std(v_roi)
        print ("V: ", v)

        
        rang_H = -0.04*s + 2.9    
        if rang_H < 0:
            rang_H = 0
        rang_S = -0.005*s + 1.02  
        
        coef_H = [1] #, 1.5, 2, 2.5]
        
        
        GT = f[:-8]+"p"+str(pap_x)+"_"+f[-8:-4]+"_GT_2EV.tif"  
        print (GT)
        img_GT = cv2.imread(GT, -1)
        image = cv2.cvtColor(img_GT, cv2.COLOR_BGR2RGB)
        print (img_GT.dtype)
        
        arr_coef_H = []
        arr_coef_S = []
        arr_F1 = []
        arr_prec = []
        arr_recall = []
        
        for hf in coef_H:
            F1_ref = 0
            F1 = 0.1
            coef_S = 1
        
            while (F1-F1_ref) > 0.01:
            #while coef_S < 1.2:
                
                if F1 < F1_ref: continue

                F1_ref = F1
                
                c1 = (h-((rang_H/2)+(h_noise/2))*hf, s-((rang_S/2)+(s_noise/2))*coef_S, 0)
                c2 = (h+((rang_H/2)+(h_noise/2))*hf, s+((rang_S/2)+(s_noise/2))*coef_S, 100)
                
                print(c1)
                print(c2)
             
        
                mask = cv2.inRange(img_hsv, c1, c2)
                result = cv2.bitwise_and(img_rgb, img_rgb, mask=mask)
                result_bgr = cv2.cvtColor(result, cv2.COLOR_RGB2BGR)
                result_bgr_8 = cv2.convertScaleAbs(result_bgr, alpha=(255.0/65535.0))
                result_rgb_8 = cv2.cvtColor(result_bgr_8, cv2.COLOR_BGR2RGB)
            
                

                FN_img = cv2.subtract(img_GT, result_bgr_8)
                FP_img = cv2.subtract(result_bgr_8, img_GT)
                TP_img = cv2.bitwise_and(result_bgr_8, img_GT)
                #TP_img = cv2.cvtColor(TP_img, cv2.COLOR_RGB2BGR)
            
                FN_img_g = cv2.cvtColor(FN_img, cv2.COLOR_BGR2GRAY)
                FP_img_g = cv2.cvtColor(FP_img, cv2.COLOR_BGR2GRAY)
                TP_img_g = cv2.cvtColor(TP_img, cv2.COLOR_BGR2GRAY)
                
                
                size = FN_img.shape
                total = size[0]*size[1]

                FN_im = FN_img_g.flatten()
                FP_im = FP_img_g.flatten()
                TP_im = TP_img_g.flatten()

                FN_list = FN_im[FN_im>5]
                FP_list = FP_im[FP_im>5]
                TP_list = TP_im[TP_im>0]

                FN = len(FN_list)
                FP = len(FP_list)
                TP = len(TP_list)

                print ("Total: ", total)
                print ("FN: ", FN)
                print ("FP: ", FP)
                print ("TP: ", TP)
                
                TN = total - TP - FP - FN

                print ("TN: ", TN)

                prec = TP / (TP+FP)
                recall = TP / (TP+FN)
                F1 = 2*(prec*recall)/(prec + recall)
                print ("CoefS:", coef_S, "F1:", F1)
            
                coef_S += 0.2
            
            coef_S = round(coef_S-0.2, 2)
            arr_prec.append(prec)
            arr_recall.append(recall)
            arr_F1.append(F1)
            arr_coef_H.append(hf)
            arr_coef_S.append(coef_S)
    
            idx += 1
            
 
            result_name = "../results/"+f[:-4]+"_p"+str(pap_x)+"_"+"coefH"+str(hf)+"coefS"+str(coef_S)+"eq+noise.png"
            cv2.imwrite (result_name, result_bgr_8)

            fig = plt.figure(figsize = (40,15))
            plt.subplot(1, 2, 1)
            plt.imshow(TP_img)
            plt.subplot(1,2,2)
            plt.imshow(result_rgb_8)
            plt.show()

            df.at[idx, "Image"] = f
            df.at[idx, "Scene"] = f[-15:-9]
            df.at[idx, "Paper"] = pap_x
            df.at[idx, "H"] = h
            df.at[idx, "S"] = s
            df.at[idx, "V"] = v
            df.at[idx, "H noise"] = h_noise
            df.at[idx, "S noise"] = s_noise
            df.at[idx, "coef H"] = hf
            df.at[idx, "H range eq"] = rang_H
            df.at[idx, "H range apl"] = (rang_H+h_noise)*hf
            df.at[idx, "Coef S"] = coef_S
            df.at[idx, "S range eq"] = rang_S
            df.at[idx, "S range apl"] = (rang_S+s_noise)*coef_S
            df.at[idx, "TP"] = TP
            df.at[idx, "FP"] = FP
            df.at[idx, "FN"] = FN
            df.at[idx, "Precision"] = prec
            df.at[idx, "Recall"] = recall
            df.at[idx, "F1"] = F1
        
        
        F1max = max(arr_F1)
        print ("F1 max:", F1max)
        pos = arr_F1.index(F1max)
        print ("position F1max:", pos)
        prec_max = arr_prec[pos]
        recall_max = arr_recall[pos]
        coef_H_max = arr_coef_H[pos]
        print ("Coef H max:", coef_H_max)
        coef_S_max = arr_coef_S[pos]
        print ("Coef S max:", coef_S_max)
        
            
print(df)
df.to_excel("../results/stats_Deteccio_Gamma1_H-S_coefSiterative.xlsx") 

In [ ]:
#DETECTION - SENSOR-DISPLAY APPROACH - Coef_S iterative


import matplotlib.pyplot as plt

folder = "Gamma_22"

px = [420, 550, 690, 850]
y = 850
winsz = 10
color_roi = (255,0,0)


files = [folder+"/papers_colors_v3_small_scene1_0347.png", folder+"/papers_colors_v3_small_scene2_0354.png", folder+"/papers_colors_v3_small_scene3_0364.png", folder+"/papers_colors_v3_small_scene4_0371.png"]

df = pd.DataFrame() 
idx = 0

for f in files:
    
    if "coef" in f or "GT" in f: continue
                
    print ("converting to HSV: "+f)
    
    img = cv2.imread(f)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    size = img.shape
    img_hsv = np.zeros((size[0], size[1], 3))
    
    
    for i in range (img.shape[0]):
        for j in range (img.shape[1]):
            R = int(img_rgb[i,j,0])
            G = int(img_rgb[i,j,1])
            B = int(img_rgb[i,j,2])
            
            RGB = (R,G,B)
            #print (RGB)
            
            maxi = max(RGB)
            mini = min(RGB)
            
            if maxi == mini:
                mini = maxi-1
            
            if maxi == RGB[0]:
                H = 60*((G-B)/(maxi-mini))
            elif maxi == RGB[1]:
                H = 120+60*((B-R)/(maxi-mini))
            elif maxi == RGB[2]:
                H = 240+60*((RGB[0]-RGB[1])/(maxi-mini)) 
                
            if maxi == 0:
                S = 0
            else:
                #s = 100*((maxi-mini)/maxi) 
                S = 100*(float((maxi-mini)/maxi))
            
            V = (maxi/255)*100  
            
            img_hsv[i,j,2] = V
            img_hsv[i,j,1] = S
            img_hsv[i,j,0] = H

 
    for x in px:
        
        if x == px[0]: pap_x = 1
        elif x == px[1]: pap_x = 2
        elif x == px[2]: pap_x = 3
        elif x == px[3]: pap_x = 4
        
        winsz = 10
        color_roi = (255,0,0)
        #img_vis = (img/256).astype('uint8')
        img_roi = cv2.rectangle(img, (y-(winsz//2), x-(winsz//2)), (y+winsz, x+winsz), color_roi, 3)
        img_roi_rgb = cv2.cvtColor(img_roi, cv2.COLOR_BGR2RGB)
        
        h_roi_np = img_hsv[(x-(winsz//2)):(x+(winsz//2)),(y-(winsz//2)):(y+(winsz//2)),0]

        
        #Descartar outliers amb mètode Chevyshev
        h_roi = h_roi_np.flatten()
        h_mean = np.mean(h_roi)
        h_std = np.std(h_roi)
        #print (h_roi)
        
        k = 3.16  #equival a p = 0.1 
        outlier_indices = np.where((h_roi > (h_mean+k*h_std)) | (h_roi < (h_mean-k*h_std)))[0]
        print ("H outliers:",  outlier_indices)
        h_no_out = np.delete(h_roi, outlier_indices)

        h = np.mean(h_no_out)
        min_h = np.min(h_no_out)
        max_h = np.max(h_no_out)
        h_noise = max_h-min_h   #variacions de saturació per la textura
        print ("H: ", h)
        print ("Std Dev H: ", h_std)
        print ("H range in roi: ", min_h,     max_h)
                
        s_roi_np = img_hsv[(x-(winsz//2)):(x+(winsz//2)),(y-(winsz//2)):(y+(winsz//2)),1]
        
        #Descartar outliers amb mètode Chevyshev
        s_roi = s_roi_np.flatten()
        s_mean = np.mean(s_roi)
        s_std = np.std(s_roi)
        
        k = 3.16  #equival a p = 0.1 
        outlier_indices = np.where((s_roi > (s_mean+k*s_std)) | (s_roi < (s_mean-k*s_std)))[0]
        print ("S outliers:",  outlier_indices)
        s_no_out = np.delete(s_roi, outlier_indices)
        
        s = np.mean(s_no_out)
        std_s = np.std(s_no_out)
        min_s = np.min(s_no_out)
        max_s = np.max(s_no_out)
        s_noise = max_s-min_s  
        print ("S: ", s)
        print ("S range in roi: ", min_s,     max_s)

        v_roi = img_hsv[(x-(winsz//2)):(x+(winsz//2)),(y-(winsz//2)):(y+(winsz//2)),2]
        v = np.mean(v_roi)
        std_v = np.std(v_roi)
        print ("V: ", v)

        
        rang_H = -0.08*s + 6.88  
        rang_S = 0.12*s + 0.56 
        
        coef_H = 1
        
        GT = f[:-8]+"p"+str(pap_x)+"_"+f[-8:-4]+"_GT_2EV.png"  
        print (GT)
        img_GT = cv2.imread(GT)
        image = cv2.cvtColor(img_GT, cv2.COLOR_BGR2RGB)
        
        arr_coef_H = []
        arr_coef_S = []
        arr_F1 = []
        arr_prec = []
        arr_recall = []
        
        F1_ref = 0
        F1 = 0.1
        coef_S = 1
        
        while (F1-F1_ref) > 0.01:
        #while coef_S < 1.2:
                
            if F1 < F1_ref: continue
                
            F1_ref = F1

            c1 = (h-(abs(rang_H/2)+(h_noise/2))*hf, s-(rang_S/3)-(s_noise/2), 0) #per sota només resto 1/3 part del rang_S (perquè no estic mesurant a la zona més lluminosa) i el noise perquè rang S és assimètric
            c2 = (h+(abs(rang_H/2)+(h_noise/2))*hf, s+((rang_S)+(s_noise/2))*coef_S, 100)
            print(c1)
            print(c2)
        
            mask = cv2.inRange(img_hsv, c1, c2)
            result = cv2.bitwise_and(img_rgb, img_rgb, mask=mask)
            result_bgr = cv2.cvtColor(result, cv2.COLOR_RGB2BGR)


            FN_img = cv2.subtract(img_GT, result_bgr)
            FP_img = cv2.subtract(result_bgr, img_GT)
            TP_img = cv2.bitwise_and(result_bgr, img_GT)
            
            FN_img_g = cv2.cvtColor(FN_img, cv2.COLOR_BGR2GRAY)
            FP_img_g = cv2.cvtColor(FP_img, cv2.COLOR_BGR2GRAY)
            TP_img_g = cv2.cvtColor(TP_img, cv2.COLOR_BGR2GRAY)
                
                
            size = FN_img.shape
            total = size[0]*size[1]

            FN_im = FN_img_g.flatten()
            FP_im = FP_img_g.flatten()
            TP_im = TP_img_g.flatten()

            FN_list = FN_im[FN_im>30]
            FP_list = FP_im[FP_im>30]
            TP_list = TP_im[TP_im>0]

            FN = len(FN_list)
            FP = len(FP_list)
            TP = len(TP_list)

            print ("Total: ", total)
            print ("FN: ", FN)
            print ("FP: ", FP)
            print ("TP: ", TP)
                
            TN = total - TP - FP - FN

            print ("TN: ", TN)

            prec = TP / (TP+FP)
            recall = TP / (TP+FN)
            F1 = 2*(prec*recall)/(prec + recall)
            print ("CoefS:", coef_S, "F1:", F1)
            
            coef_S += 0.2
            
        coef_S = round(coef_S - 0.2, 2)
        arr_prec.append(prec)
        arr_recall.append(recall)
        arr_F1.append(F1)
        arr_coef_H.append(hf)
        arr_coef_S.append(coef_S)
    
        idx += 1
            
        result_name = "../results/"+f[:-4]+"p"+str(pap_x)+"_"+"coefH"+str(hf)+"coefS"+str(coef_S)+"eq+noise.png"
        cv2.imwrite (result_name, result_bgr)

        fig = plt.figure(figsize = (40,15))
        plt.subplot(1, 2, 1)
        plt.imshow(TP_img)
        plt.subplot(1,2,2)
        plt.imshow(result)
        plt.show()

        df.at[idx, "Image"] = f
        df.at[idx, "Scene"] = f[-15:-9]
        df.at[idx, "Paper"] = pap_x
        df.at[idx, "H"] = h
        df.at[idx, "S"] = s
        df.at[idx, "V"] = v
        df.at[idx, "H noise"] = h_noise
        df.at[idx, "S noise"] = s_noise
        df.at[idx, "coef H"] = hf
        df.at[idx, "H range eq"] = rang_H
        df.at[idx, "H range apl"] = (rang_H+h_noise)*coef_H
        df.at[idx, "Coef S"] = coef_S
        df.at[idx, "S range eq"] = rang_S
        df.at[idx, "S range apl"] = (rang_S+s_noise/2)*coef_S+rang_S/3+s_noise/2
        df.at[idx, "TP"] = TP
        df.at[idx, "FP"] = FP
        df.at[idx, "FN"] = FN
        df.at[idx, "Precision"] = prec
        df.at[idx, "Recall"] = recall
        df.at[idx, "F1"] = F1
        
        
    F1max = max(arr_F1)
    print ("F1 max:", F1max)
    pos = arr_F1.index(F1max)
    print ("position F1max:", pos)
    prec_max = arr_prec[pos]
    recall_max = arr_recall[pos]
    coef_H_max = arr_coef_H[pos]
    print ("Coef H max:", coef_H_max)
    coef_S_max = arr_coef_S[pos]
    print ("Coef S max:", coef_S_max)
        
            
print(df)
df.to_excel("../results/stats_Deteccio_Gamma22_H-S_coefSiterative.xlsx") 

In [ ]:
#DETECTION - PERCEPTUAL APPROACH - Coef_C iterative

import matplotlib.pyplot as plt
import colour

folder = "Perceptual"

px = [420, 550, 690, 850]
y = 850
winsz = 10
color_roi = (255,0,0)


files = [folder+"/papers_colors_v3_small_scene1_0347.png", folder+"/papers_colors_v3_small_scene2_0354.png", folder+"/papers_colors_v3_small_scene3_0364.png", folder+"/papers_colors_v3_small_scene4_0371.png"]


df = pd.DataFrame() 
idx = 0


def ProPhoto_Lab2LCH (image):
    im = cv2.imread(image)
    im_RGB = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
    illuminant_RGB = [0.34570, 0.35850]  #D50
    illuminant_XYZ = [0.34570, 0.35850]  #D50

    PROPHOTO_RGB_TO_XYZ_MATRIX = ([[ 7.97667235e-01, 1.35192231e-01, 3.13525290e-02], [ 2.88037454e-01, 7.11876883e-01, 8.56626476e-05], [ 0.00000000e+00, 0.00000000e+00, 8.25188285e-01]])
    chromatic_adaptation_transform = 'Bradford'
    decoding = colour.models.cctf_decoding_ProPhotoRGB
    print (decoding)

    im_XYZ = colour.RGB_to_XYZ(
         im_RGB/255,
         illuminant_RGB,
         illuminant_XYZ,
         PROPHOTO_RGB_TO_XYZ_MATRIX,
         chromatic_adaptation_transform,
         cctf_decoding=decoding)
         

    im_LAB = colour.XYZ_to_Lab(im_XYZ, illuminant = (0.34570, 0.35850))
    size = im_LAB.shape

    im_LCH = np.zeros((size[0], size[1], 3))

    for i in range (im_LCH.shape[0]):
        for j in range (im_LCH.shape[1]):
            im_LCH[i,j,0] = im_LAB[i,j,0]
            im_LCH[i,j,1] = math.sqrt((im_LAB[i,j,1])**2+(im_LAB[i,j,2])**2)
            if im_LAB[i,j,2] >= 0 and im_LAB[i,j,1] >= 0:
                im_LCH[i,j,2] = math.degrees(math.atan((im_LAB[i,j,2])/(im_LAB[i,j,1])))
            elif im_LAB[i,j,2] >= 0 and im_LAB[i,j,1] < 0:
                im_LCH[i,j,2] = math.degrees(math.atan((im_LAB[i,j,2])/(im_LAB[i,j,1])))+180
            elif im_LAB[i,j,2] < 0 and im_LAB[i,j,1] < 0:
                im_LCH[i,j,2] = math.degrees(math.atan((im_LAB[i,j,2])/(im_LAB[i,j,1])))-180
            else:
                im_LCH[i,j,2] = math.degrees(math.atan((im_LAB[i,j,2])/(im_LAB[i,j,1])))
    
    return (im_LCH)


for f in files:
    if "coef" in f or "GT" in f: continue
    
    for x in px:

        if x == px[0]: pap_x = 1
        elif x == px[1]: pap_x = 2
        elif x == px[2]: pap_x = 3
        elif x == px[3]: pap_x = 4
        
        img_lch = ProPhoto_Lab2LCH (f)


        h_roi_np = img_lch[(x-(winsz//2)):(x+(winsz//2)),(y-(winsz//2)):(y+(winsz//2)),2]

        h_roi = h_roi_np.flatten()
        h_mean = np.mean(h_roi)
        h_std = np.std(h_roi)
        
        k = 3.16  #equival a p = 0.1 
        outlier_indices = np.where((h_roi > (h_mean+k*h_std)) | (h_roi < (h_mean-k*h_std)))[0]
        print ("H outliers:",  outlier_indices)
        h_no_out = np.delete(h_roi, outlier_indices)

        h = np.mean(h_no_out)
        min_h = np.min(h_no_out)
        max_h = np.max(h_no_out)
        h_noise = max_h-min_h  
        print ("H: ", h)
        print ("Std Dev H: ", h_std)
        print ("H range in roi: ", min_h,     max_h)

        c_roi_np = img_lch[(x-(winsz//2)):(x+(winsz//2)),(y-(winsz//2)):(y+(winsz//2)),1]
        
        #Descartar outliers amb mètode Chevyshev
        c_roi = c_roi_np.flatten()
        c_mean = np.mean(c_roi)
        c_std = np.std(c_roi)
        
        k = 3.16  #equival a p = 0.1 
        outlier_indices = np.where((c_roi > (c_mean+k*c_std)) | (c_roi < (c_mean-k*c_std)))[0]
        print ("S outliers:",  outlier_indices)
        c_no_out = np.delete(c_roi, outlier_indices)
        
        c = np.mean(c_no_out)
        std_c = np.std(c_no_out)
        min_c = np.min(c_no_out)
        max_c = np.max(c_no_out)
        c_noise = max_c-min_c   
        
        l_roi = img_lch[(x-(winsz//2)):(x+(winsz//2)),(y-(winsz//2)):(y+(winsz//2)),0]
        l = np.mean(l_roi)
        std_l = np.std(l_roi)
        print ("L: ", l)

        rang_H = abs(-0.17*c + 11)  
        rang_C = 0.25*c + 0.58
        
        coef_C = 1   
        coef_H = 1
        F1_ref = 0
        F1 = 0.1
        
        
        GT = f[:-8]+"p"+str(pap_x)+"_"+f[-8:-4]+"_GT_2EV.png" 
        print (GT)
        img_GT = cv2.imread(GT)
        image = cv2.cvtColor(img_GT, cv2.COLOR_BGR2RGB)
        
        while (F1-F1_ref) > 0.01:

            if F1 < F1_ref: continue

            F1_ref = F1
            color1 = (0, c-(rang_C+c_noise/2)*coef_C, h-(rang_H/2+h_noise/2)*coef_H)
            color2 = (255, c+rang_C/2+c_noise/2, h+(rang_H/2+h_noise/2)*coef_H)
            
            image_ref = cv2.imread(f)
            image_rgb = cv2.cvtColor(image_ref, cv2.COLOR_BGR2RGB)
            image_roi = cv2.rectangle(image_rgb, (y-(winsz//2), x-(winsz//2)), (y+winsz, x+winsz), color_roi, 3)
            
            mask = np.zeros((int(img_lch.shape[0]), int(img_lch.shape[1])), dtype = np.int8)

            for i in range (mask.shape[0]):
                    for j in range (mask.shape[1]):
                        if color1[2] < img_lch[i,j,2] < color2[2] and color1[1] < img_lch[i,j,1] < color2[1]:
                            mask[i,j]=int(1)
            
            
            result = cv2.bitwise_and(image_ref, image_ref, mask=mask)
            result_bgr = cv2.cvtColor(result, cv2.COLOR_RGB2BGR)

            FN_img = cv2.subtract(img_GT, result)
            FP_img = cv2.subtract(result, img_GT)
            TP_img = cv2.bitwise_and(result, img_GT)
            
            FN_img_g = cv2.cvtColor(FN_img, cv2.COLOR_BGR2GRAY)
            FP_img_g = cv2.cvtColor(FP_img, cv2.COLOR_BGR2GRAY)
            TP_img_g = cv2.cvtColor(TP_img, cv2.COLOR_BGR2GRAY)
                
            size = FN_img.shape
            total = size[0]*size[1]

            FN_im = FN_img_g.flatten()
            FP_im = FP_img_g.flatten()
            TP_im = TP_img_g.flatten()

            FN_list = FN_im[FN_im>0]
            FP_list = FP_im[FP_im>0]
            TP_list = TP_im[TP_im>0]

            FN = len(FN_list)
            FP = len(FP_list)
            TP = len(TP_list)

            print ("Total: ", total)
            print ("FN: ", FN)
            print ("FP: ", FP)
            print ("TP: ", TP)
                
            TN = total - TP - FP - FN

            print ("TN: ", TN)

            prec = TP / (TP+FP)
            recall = TP / (TP+FN)
            F1 = 2*(prec*recall)/(prec + recall)
            
            coef_C += 0.2
            
        if F1 < F1_ref:
            F1 = F1_ref
            coef_C = round(coef_H - 0.4, 2)
            print ("CoefH:", coef_H, "F1:", F1)

        else:
            coef_C = round(coef_H - 0.2, 2)
            print ("CoefH:", coef_H, "F1:", F1)
        
        df.at[idx, "Image"] = f
        df.at[idx, "Scene"] = f[-15:-9]
        df.at[idx, "Paper"] = pap_x
        df.at[idx, "H"] = h
        df.at[idx, "C"] = c
        df.at[idx, "L"] = l
        df.at[idx, "H noise"] = h_noise
        df.at[idx, "C noise"] = c_noise
        df.at[idx, "coef H"] = coef_H
        df.at[idx, "H range eq"] = rang_H
        df.at[idx, "H range"] = (rang_H+h_noise)*coef_H
        df.at[idx, "Coef C"] = coef_C
        df.at[idx, "S range eq"] = rang_C
        df.at[idx, "C range"] = (rang_C+c_noise/2)*coef_C+rang_C/2+c_noise/2
        df.at[idx, "TP"] = TP
        df.at[idx, "FP"] = FP
        df.at[idx, "FN"] = FN
        df.at[idx, "Precision"] = prec
        df.at[idx, "Recall"] = recall
        df.at[idx, "F1"] = F1

        
        idx += 1

 
        result_name = "../results/"+f[:-4]+"p"+str(pap_x)+"_"+"coefH"+str(coef_H)+"coefC"+str(coef_C)+"_v2.png"
        cv2.imwrite (result_name, result)

        fig = plt.figure(figsize = (40,15))
        plt.subplot(1, 2, 1)
        plt.imshow(image_roi)
        plt.subplot(1,2,2)
        plt.imshow(result_bgr)
        plt.show()

print(df)
df.to_excel("../results/stats_Deteccio_Per_C-H_coefHiterative.xlsx")    